# OmniVoice Project Studio — Kaggle

Notebook này chạy `omnivoice-project-studio` trên Kaggle và có thêm **FRPC self-heal** để tránh lỗi:

```text
Could not create share link
PermissionError: [Errno 13] Permission denied
```

Nguyên nhân đã xác định: binary `frpc` của Gradio có thể tồn tại trong cache nhưng bị mất quyền execute. Cell kiểm tra bên dưới sẽ tự sửa quyền trước khi tạo public Gradio link.


In [ ]:
# 1. Kiểm tra GPU / môi trường
import os, sys, subprocess

print("Python:", sys.version)
print("Kaggle:", os.path.exists("/kaggle"))
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# 2. Clone / cập nhật repo OmniVoice của bạn
REPO_URL = "https://github.com/binhminhanh1235/OmniVoice.git"
REPO_DIR = "/kaggle/working/OmniVoice"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo đã tồn tại, cập nhật master...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/master"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)


In [ ]:
# 3. Cài OmniVoice
# Nếu môi trường Kaggle hiện tại đã cài đúng dependency, có thể chạy lại cell này an toàn.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import gradio as gr
print("Gradio:", gr.__version__)


In [ ]:
# 4. Cấu hình Project Studio
TTS_DEVICE = "cuda"
ASR_MODEL = "openai/whisper-small"
ASR_DEVICE = "cpu"
WORKSPACE = "/kaggle/working/OmniVoiceStudio"

os.makedirs(WORKSPACE, exist_ok=True)

print("TTS_DEVICE =", TTS_DEVICE)
print("ASR_MODEL  =", ASR_MODEL)
print("ASR_DEVICE =", ASR_DEVICE)
print("WORKSPACE  =", WORKSPACE)


## 5. Sửa FRPC trước khi tạo share link

Gradio chỉ `chmod` binary FRPC khi file vừa được tải mới. Nếu Kaggle/cache giữ file nhưng làm mất execute bit, `launch(share=True)` có thể thất bại với `PermissionError(13)`.

Cell này:
1. Tìm đúng `BINARY_PATH` mà Gradio đang dùng.
2. Nếu file tồn tại, bật execute cho user/group/others.
3. Chạy `frpc --version` để xác nhận binary thực sự chạy được.
4. Nếu binary chưa tồn tại, Gradio sẽ tải nó khi launch, sau đó có thể chạy lại cell này nếu cần.


In [ ]:
# 5. FRPC self-heal
import os
import stat
import subprocess
from pathlib import Path

from gradio.tunneling import BINARY_PATH

frpc = Path(BINARY_PATH)
print("FRPC path:", frpc)

if frpc.exists():
    mode_before = oct(frpc.stat().st_mode)
    print("Mode before:", mode_before)
    print("Executable before:", os.access(frpc, os.X_OK))

    mode = frpc.stat().st_mode
    frpc.chmod(
        mode
        | stat.S_IXUSR
        | stat.S_IXGRP
        | stat.S_IXOTH
    )

    print("Mode after :", oct(frpc.stat().st_mode))
    print("Executable after:", os.access(frpc, os.X_OK))

    test = subprocess.run(
        [str(frpc), "--version"],
        text=True,
        capture_output=True,
    )
    print("FRPC returncode:", test.returncode)
    print("FRPC stdout:", test.stdout.strip())
    print("FRPC stderr:", test.stderr.strip())

    if test.returncode != 0:
        raise RuntimeError("FRPC tồn tại nhưng không chạy được.")
else:
    print("FRPC chưa tồn tại. Gradio sẽ tải binary khi tạo share link.")


In [ ]:
# 6. Optional: kiểm tra nhanh Gradio share tunnel
# Đặt RUN_GRADIO_SMOKE_TEST=True nếu muốn test tunnel trước khi load OmniVoice.
RUN_GRADIO_SMOKE_TEST = False

if RUN_GRADIO_SMOKE_TEST:
    import gradio as gr

    smoke = gr.Interface(
        lambda x: f"OK: {x}",
        inputs="text",
        outputs="text",
    )
    smoke.launch(
        server_name="0.0.0.0",
        server_port=7861,
        share=True,
        prevent_thread_lock=True,
        show_error=True,
    )
    print("Smoke-test share URL:", smoke.share_url)


## 7. Launch OmniVoice Project Studio

Cell dưới đây chạy đúng launcher của repo với `--share`.

Nếu Kaggle session/cache làm mất execute bit lần nữa, chỉ cần chạy lại cell **FRPC self-heal** rồi launch lại.


In [ ]:
# 7. Launch Project Studio
import shlex

cmd = [
    "omnivoice-project-studio",
    "--model", "k2-fsa/OmniVoice",
    "--device", TTS_DEVICE,
    "--workspace", WORKSPACE,
    "--asr-model", ASR_MODEL,
    "--asr-device", ASR_DEVICE,
    "--share",
]

print("Running:")
print(" ".join(shlex.quote(x) for x in cmd))

subprocess.run(cmd, check=False)


## Troubleshooting nhanh

Nếu vẫn thấy:

```text
Could not create share link
```

hãy chạy lại cell **FRPC self-heal**. Kết quả mong đợi:

```text
Executable after: True
FRPC returncode: 0
FRPC stdout: 0.44.0
```

Nếu smoke test tạo share link được nhưng Project Studio vẫn lỗi, có thể dùng diagnostic tunnel riêng để lấy exception thật thay vì thông báo chung của Gradio.
